In [2]:
import pandas as pd
import seaborn as sns
import numpy as np

df = pd.read_csv("../data/heart_disease_uci.csv")

In [3]:
df.head()

# df["num"].value_counts()

# (df["chol"] == 0).sum()

# for col in df.columns:
#     if df[col].dtype != "object":
#         print(col, (df[col] == 0).sum())


# df.corr(numeric_only=True)["num"].sort_values()

df.isnull().sum().sort_values(ascending=False)

ca          611
thal        486
slope       309
fbs          90
oldpeak      62
trestbps     59
exang        55
thalch       55
chol         30
restecg       2
cp            0
dataset       0
id            0
age           0
sex           0
num           0
dtype: int64

In [4]:
df = df.drop(columns=['id'])
df["chol"] = df["chol"].replace(0, np.nan)

X = df.drop(columns=["num"])
y = df["num"]

print(X.shape)
print(y.shape)

(920, 14)
(920,)


In [5]:
# Splitting
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y,test_size=0.3, random_state=42)

y_train.value_counts()
y_test.value_counts()

X_train.isnull().sum().sort_values(ascending=False)


ca          422
thal        333
slope       208
chol        144
fbs          67
oldpeak      43
trestbps     40
thalch       36
exang        36
restecg       2
dataset       0
cp            0
age           0
sex           0
dtype: int64

In [25]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train['sex'] = X_train['sex'].replace({"Male": 1, "Female": 0})
X_test['sex'] = X_test['sex'].replace({"Male": 1, "Female": 0})

X_train['fbs'] = X_train['fbs'].replace({True: 1, False: 0})
X_test['fbs'] = X_test['fbs'].replace({True: 1, False: 0})

X_train['exang'] = X_train['exang'].replace({True: 1, False: 0})
X_test['exang'] = X_test['exang'].replace({True: 1, False: 0})

X_train['sex'] = X_train['sex'].astype(int)
X_test['sex'] = X_test['sex'].astype(int)

X_train['fbs'] = X_train['fbs'].astype(float)
X_test['fbs'] = X_test['fbs'].astype(float)

X_train['exang'] = X_train['exang'].astype(float)
X_test['exang'] = X_test['exang'].astype(float)

In [ ]:
numerical_features = ["age", "trestbps", "chol", "thalch", "oldpeak", "ca", "sex", "fbs", "exang"]

categorical_features = ["cp", "restecg", "slope", "thal", "dataset"]

target = "num"

array([0, 1, nan], dtype=object)

In [32]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder="passthrough"
)

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])

logistic_pipeline.fit(X_train, y_train)

preds = logistic_pipeline.predict(X_test)

In [37]:
print(preds[:20])
print(y_test.iloc[:20].values)
print()

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

cm = confusion_matrix(y_train, preds)
print(cm)
print("classification_report:")
print(classification_report(y_train, preds))

[1 3 0 0 0 0 0 1 0 1 0 0 3 3 1 0 1 0 1 1]
[1 0 3 1 0 0 0 0 0 1 0 1 3 0 4 3 3 0 1 0]

[[248  30   6   3   1]
 [ 53 113   4  15   0]
 [ 12  35  13  16   0]
 [  9  21   9  35   1]
 [  1   7   0   9   3]]
classification_report:
              precision    recall  f1-score   support

           0       0.77      0.86      0.81       288
           1       0.55      0.61      0.58       185
           2       0.41      0.17      0.24        76
           3       0.45      0.47      0.46        75
           4       0.60      0.15      0.24        20

    accuracy                           0.64       644
   macro avg       0.55      0.45      0.47       644
weighted avg       0.62      0.64      0.62       644



In [51]:
# Taking binary classifiction
# 0 -> No disease
# 1, 2, 3, 4 -> Disease

y_binary = (df["num"] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y_binary, stratify=y_binary, test_size=0.3, random_state=42)

X_train = X_train.copy()
X_test = X_test.copy()

X_train['sex'] = X_train['sex'].replace({"Male": 1, "Female": 0})
X_test['sex'] = X_test['sex'].replace({"Male": 1, "Female": 0})

X_train['fbs'] = X_train['fbs'].replace({True: 1, False: 0})
X_test['fbs'] = X_test['fbs'].replace({True: 1, False: 0})

X_train['exang'] = X_train['exang'].replace({True: 1, False: 0})
X_test['exang'] = X_test['exang'].replace({True: 1, False: 0})

X_train['sex'] = X_train['sex'].astype(int)
X_test['sex'] = X_test['sex'].astype(int)

X_train['fbs'] = X_train['fbs'].astype(float)
X_test['fbs'] = X_test['fbs'].astype(float)

X_train['exang'] = X_train['exang'].astype(float)
X_test['exang'] = X_test['exang'].astype(float)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="passthrough"
)

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])

logistic_pipeline.fit(X_train, y_train)

preds = logistic_pipeline.predict(X_test)

cm = confusion_matrix(y_test, preds)
print(f"Confusion Matrix: \n{cm}\n\n")
print(classification_report(y_test, preds))

Confusion Matrix: 
[[ 98  25]
 [ 26 127]]


              precision    recall  f1-score   support

           0       0.79      0.80      0.79       123
           1       0.84      0.83      0.83       153

    accuracy                           0.82       276
   macro avg       0.81      0.81      0.81       276
weighted avg       0.82      0.82      0.82       276

